In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

In [2]:
movies = pd.read_csv("../data/movies.csv")
ratings = pd.read_csv("../data/ratings.csv")
tags = pd.read_csv("../data/tags.csv")

In [3]:
print("Movies:", movies.shape)
print("Ratings:", ratings.shape)
print("Tags:", tags.shape)

Movies: (87585, 3)
Ratings: (32000204, 4)
Tags: (2000072, 4)


In [4]:
movies.isnull().sum()

movieId    0
title      0
genres     0
dtype: int64

In [5]:
ratings.isnull().sum()

userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

In [6]:
tags.isnull().sum()

userId        0
movieId       0
tag          17
timestamp     0
dtype: int64

In [7]:
tags = tags.dropna(subset=["tag"])

In [8]:
movie_tags = (
    tags.groupby("movieId")["tag"]
    .apply(lambda x: " ".join(x.astype(str)))
    .reset_index()
)

movie_tags.head()

,movieId,tag
0,1,children Disney animation children Disney Disn...
1,2,Robin Williams fantasy Robin Williams time tra...
2,3,comedinha de velhinhos engraÃƒÂ§ada comedinha ...
3,4,characters slurs based on novel or book chick ...
4,5,Fantasy pregnancy remake family Steve Martin s...


In [9]:
movies = movies.merge(
    movie_tags,
    on="movieId",
    how="left"
)

movies["tag"] = movies["tag"].fillna("")

In [10]:
movie_ratings = ratings.groupby("movieId").agg(
    avg_rating=("rating", "mean"),
    num_ratings=("rating", "count")
).reset_index()

movie_ratings.head()

,movieId,avg_rating,num_ratings
0,1,3.897438,68997
1,2,3.275758,28904
2,3,3.139447,13134
3,4,2.845331,2806
4,5,3.059602,13154


In [11]:
movies = movies.merge(
    movie_ratings,
    on="movieId",
    how="left"
)

# Left join introduces NaN for movies with zero ratings.
# Fill and cast so num_ratings stays a clean integer column.
movies["num_ratings"] = movies["num_ratings"].fillna(0).astype(int)
movies["avg_rating"] = movies["avg_rating"].fillna(0.0)

movies.head()

,movieId,title,genres,tag,avg_rating,num_ratings
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,children Disney animation children Disney Disn...,3.897438,68997
1,2,Jumanji (1995),Adventure|Children|Fantasy,Robin Williams fantasy Robin Williams time tra...,3.275758,28904
2,3,Grumpier Old Men (1995),Comedy|Romance,comedinha de velhinhos engraÃƒÂ§ada comedinha ...,3.139447,13134
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,characters slurs based on novel or book chick ...,2.845331,2806
4,5,Father of the Bride Part II (1995),Comedy,Fantasy pregnancy remake family Steve Martin s...,3.059602,13154


In [12]:
movies["content"] = (
    movies["genres"] +
    " " +
    movies["tag"]
)

In [13]:
tfidf = TfidfVectorizer(
    stop_words="english"
)

tfidf_matrix = tfidf.fit_transform(
    movies["content"]
)

In [14]:
tfidf_matrix.shape

(87585, 46657)

In [15]:
indices = pd.Series(
    movies.index,
    index=movies["title"]
)
indices = indices[~indices.index.duplicated(keep="first")]

In [16]:
def search_movie(query):
    return movies[
        movies["title"].str.contains(
            query,
            case=False,
            na=False
        )
    ][["title"]].head(10)

In [17]:
search_movie("matrix")

,title
2480,"Matrix, The (1999)"
6248,"Matrix Reloaded, The (2003)"
6810,"Matrix Revolutions, The (2003)"
9286,"Animatrix, The (2003)"
28908,Return to Source: The Philosophy of The Matrix...
40032,Armitage: Dual Matrix (2002)
46862,The Matrix Revisited (2001)
50298,The Living Matrix (2009)
51063,Matrix of Evil (2003)
67243,Sex and the Matrix (2000)


In [18]:
def recommend(movie_title, n=10, min_ratings=50):

    if movie_title not in indices:
        return f"Movie '{movie_title}' not found."

    idx = indices[movie_title]

    sim_scores = linear_kernel(
        tfidf_matrix[idx],
        tfidf_matrix
    ).flatten()

    # Fetch extra candidates to account for the min_ratings filter below.
    # n*5 gives ample buffer before we trim down to n.
    sim_indices = sim_scores.argsort()[-(n * 5 + 1):][::-1]

    # Remove the query movie itself by position, not by slice [1:].
    # [1:] is fragile when two movies share sim=1.0 — argsort isn't stable.
    sim_indices = sim_indices[sim_indices != idx]

    candidates = movies.iloc[sim_indices][[
        "title",
        "genres",
        "avg_rating",
        "num_ratings"
    ]].copy()

    candidates["similarity"] = sim_scores[sim_indices]

    # Drop low-vote noise (replaces the dead movies_filtered block).
    candidates = candidates[
        candidates["num_ratings"] >= min_ratings
    ]

    return candidates.sort_values(
        by=["similarity", "avg_rating"],
        ascending=False
    ).head(n)

In [19]:
recommend("Toy Story (1995)")  # Test-1

,title,genres,avg_rating,num_ratings,similarity
3021,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,3.812043,32683,0.887562
2264,"Bug's Life, A (1998)",Adventure|Animation|Children|Comedy,3.558105,26736,0.801455
14815,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX,3.827643,20327,0.705850
4781,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,3.837442,46036,0.696438
39850,Finding Dory (2016),Adventure|Animation|Comedy,3.537607,5624,0.695902
6259,Finding Nemo (2003),Adventure|Animation|Children|Comedy,3.816500,46128,0.690139
8248,"Incredibles, The (2004)",Action|Adventure|Animation|Children|Comedy,3.848238,41463,0.680559
26634,The Adventures of André and Wally B. (1984),Animation,2.115385,52,0.666663
19879,Monsters University (2013),Adventure|Animation|Comedy,3.462654,6587,0.651931
18314,Knick Knack (1989),Animation|Children,3.476945,347,0.649267


In [20]:
recommend("Matrix, The (1999)")  # Test-2

,title,genres,avg_rating,num_ratings,similarity
6248,"Matrix Reloaded, The (2003)",Action|Adventure|Sci-Fi|Thriller|IMAX,3.372616,29835,0.858926
6810,"Matrix Revolutions, The (2003)",Action|Adventure|Sci-Fi|Thriller|IMAX,3.236170,23699,0.812110
2015,Tron (1982),Action|Adventure|Sci-Fi,3.334628,11973,0.552418
1678,Dark City (1998),Adventure|Film-Noir|Sci-Fi|Thriller,3.804360,14404,0.539876
2580,"Thirteenth Floor, The (1999)",Drama|Sci-Fi|Thriller,3.365052,4850,0.522781
6668,Avalon (2001),Drama|Fantasy|Sci-Fi,3.416667,744,0.516167
2509,eXistenZ (1999),Action|Sci-Fi|Thriller,3.366370,6619,0.480814
1014,"Lawnmower Man, The (1992)",Action|Horror|Sci-Fi|Thriller,2.744391,7443,0.473357
15677,Tron: Legacy (2010),Action|Adventure|Sci-Fi|IMAX,3.287812,7458,0.453913
50259,OtherLife (2017),Crime|Mystery|Sci-Fi,3.240964,166,0.442571


In [21]:
recommend("Fight Club (1999)")  # Test-3

,title,genres,avg_rating,num_ratings,similarity
618,Primal Fear (1996),Crime|Drama|Mystery|Thriller,3.768657,12797,0.589425
46,Seven (a.k.a. Se7en) (1995),Mystery|Thriller,4.087199,63298,0.559926
4773,Donnie Darko (2001),Drama|Mystery|Sci-Fi|Thriller,3.935533,35452,0.538756
2238,American History X (1998),Crime|Drama,4.130444,38967,0.513634
14338,Shutter Island (2010),Drama|Mystery|Thriller,3.993126,29096,0.502895
8237,The Machinist (2004),Drama|Mystery|Thriller,3.810697,12200,0.490383
1566,"Game, The (1997)",Drama|Mystery|Thriller,3.874517,23310,0.476657
2670,"Sixth Sense, The (1999)",Drama|Horror|Mystery,4.003210,57010,0.468095
4123,Memento (2000),Mystery|Thriller,4.141601,53658,0.452563
2905,Being John Malkovich (1999),Comedy|Drama|Fantasy,3.933888,36151,0.449897


In [22]:
recommend("Monsters, Inc. (2001)")  # Test-4

,title,genres,avg_rating,num_ratings,similarity
2264,"Bug's Life, A (1998)",Adventure|Animation|Children|Comedy,3.558105,26736,0.784321
3021,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,3.812043,32683,0.723568
0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,3.897438,68997,0.696438
8248,"Incredibles, The (2004)",Action|Adventure|Animation|Children|Comedy,3.848238,41463,0.694471
19879,Monsters University (2013),Adventure|Animation|Comedy,3.462654,6587,0.678565
6259,Finding Nemo (2003),Adventure|Animation|Children|Comedy,3.816500,46128,0.670944
39850,Finding Dory (2016),Adventure|Animation|Comedy,3.537607,5624,0.662672
10812,Cars (2006),Animation|Children|Comedy,3.314706,11703,0.649266
18315,For the Birds (2000),Animation|Children|Comedy,3.934012,1720,0.625276
18314,Knick Knack (1989),Animation|Children,3.476945,347,0.621890


In [23]:
movies.to_csv(
    "../outputs/content_based_movies.csv",
    index=False
)